# Stream scaler demo (PYNQ-Z2)

In [ ]:
from pynq import Overlay, allocate, MMIO
import numpy as np
ol = Overlay('design_1.bit')
dma = ol.axi_dma_0
leds = ol.axi_gpio_0

In [ ]:
# gain = 2.0 in Q8.8 on GPIO channel 2
leds.channel2.write(0x0200, 0xFFFF)
leds.channel1.write(0b1111, 0xF)

In [ ]:
n = 1024
src = allocate(shape=(n,), dtype=np.uint32)
dst = allocate(shape=(n,), dtype=np.uint32)
src[:] = np.arange(n)
dma.sendchannel.transfer(src)
dma.recvchannel.transfer(dst)
dma.sendchannel.wait(); dma.recvchannel.wait()
assert (dst[:8] == 2 * src[:8]).all()

In [ ]:
regs = MMIO(0x40400000, 0x10000)
print(hex(regs.read(0x04)))  # MM2S_DMASR